# 04 MLflow tracing, guardrails, and evaluation

## Learning objectives

- verify that retriever evidence has MLflow's required document shape;
- separate deterministic policy checks from LLM judges;
- compare baseline and change in one stable experiment;
- fail a release on critical access, action, abstention, or citation errors.


In [ ]:
# Notebook preflight — configuration only; this cell makes no cloud request.
import importlib.util
import sys
from pathlib import Path

setup_path = next(
    path
    for parent in (Path.cwd(), *Path.cwd().parents)
    for path in (
        parent / "notebook_setup.py",
        parent / "examples" / "agentic-ops-rag" / "notebook_setup.py",
    )
    if path.is_file()
)
spec = importlib.util.spec_from_file_location("agentic_ops_rag_setup", setup_path)
setup = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = setup
spec.loader.exec_module(setup)
course_root = setup.find_course_root(setup_path.parent)
session = setup.prepare_notebook_environment(course_root)
session.safe_summary()


## Trace shape before judge quality

Retrieval judges inspect `RETRIEVER` span outputs. If those outputs omit
`page_content`, `doc_uri`, or `chunk_id`, groundedness cannot see the evidence.
The SDK adapters normalize both Azure AI Search and Databricks AI Search results
to the same MLflow document contract.


In [ ]:
from aai_core.rag import mlflow_documents

pipeline = session.offline_pipeline()
search_results = pipeline.retriever.search(
    "Explain ERR-PAY-503",
    mode="hybrid",
    top_k=3,
    filters={"tenant_id": "tenant-alpha", "region": "eastus"},
    provider_options={"allowed_groups": ("ops-payments",)},
)
documents = mlflow_documents(search_results)
assert all("page_content" in document for document in documents)
assert all("doc_uri" in document["metadata"] for document in documents)
assert all("chunk_id" in document["metadata"] for document in documents)
documents


## Deterministic checks first

Tenant isolation, secret refusal, citations, schema validation, exact tool
allowlists, region and group authorization, current-evidence selection, and
human approval are code-level policies. LLM judges complement them with
retrieval relevance, sufficiency, and groundedness; an experimental judge is
never the sole safety gate.


In [ ]:
from agentic_ops_rag import RetrievalMode
from agentic_ops_rag.evaluation import benchmark, load_cases, release_gate

cases = load_cases(course_root / "data" / "evaluation_cases.jsonl")
offline_metrics = benchmark(
    pipeline,
    cases,
    mode=RetrievalMode.HYBRID,
)
offline_gate = release_gate(offline_metrics)
offline_gate.model_dump(mode="json")


In [ ]:
# YOUR TURN — TODO: classify every gate metric as deterministic or judge-based.
metric_owner = {
    "security/tenant_isolation": "deterministic",
    "security/region_isolation": "deterministic",
    "security/group_authorization": "deterministic",
    "security/current_evidence": "deterministic",
    "safety/action_approval": "deterministic",
    "answer/citation_integrity": "deterministic",
    "retrieval_groundedness/mean": "llm_judge",
    "retrieval_sufficiency/mean": "llm_judge",
}
metric_owner


In [ ]:
# CHECK YOUR WORK
assert metric_owner["security/tenant_isolation"] == "deterministic"
assert metric_owner["security/region_isolation"] == "deterministic"
assert metric_owner["security/group_authorization"] == "deterministic"
assert metric_owner["safety/action_approval"] == "deterministic"
assert metric_owner["retrieval_groundedness/mean"] == "llm_judge"
"Hard policies do not depend on a probabilistic judge."


In [ ]:
# Reference solution
critical_deterministic_metrics = {
    name for name, owner in metric_owner.items() if owner == "deterministic"
}
assert {
    "security/tenant_isolation",
    "security/region_isolation",
    "security/group_authorization",
    "security/current_evidence",
    "safety/action_approval",
    "answer/citation_integrity",
}.issubset(critical_deterministic_metrics)


## Optional local MLflow evidence

Enable this cell when you want a repository-local SQLite run. It uses one trace
owner (`aai-core` SDK spans), a governed experiment, and baseline/change
metadata. Do not enable OpenAI or LangChain autologging for the same calls or
you will duplicate provider spans and token evidence.


In [ ]:
RUN_MLFLOW = False
mlflow_run_id = None
if RUN_MLFLOW:
    import mlflow

    from aai_core.experiments import ExperimentRunMetadata, RunPurpose
    from aai_core.tracing import TraceIntegration

    tracking_uri = f"sqlite:///{course_root / '.aai' / 'mlflow.db'}"
    (course_root / ".aai").mkdir(parents=True, exist_ok=True)
    session.context.configure_tracing(
        tracking_uri=tracking_uri,
        integration=TraceIntegration.SDK,
    )
    with session.context.experiments.run(
        run_name="hybrid-retrieval-offline-result",
        metadata=ExperimentRunMetadata(
            purpose=RunPurpose.RESULT,
            change_id="ops-rag-hybrid-v1",
            change_summary="Compare hybrid retrieval with the fixed cases",
        ),
        parameters={"measurement_source": "simulated_offline_fixture"},
    ) as active_run:
        mlflow.log_metrics(offline_metrics)
        mlflow.set_tag("aai.gate_passed", str(offline_gate.passed).lower())
        mlflow_run_id = active_run.info.run_id
mlflow_run_id


## Optional connected MLflow GenAI evaluation

`mlflow.genai.evaluate()` is distinct from classic model evaluation. The RAG
judges need real traces; RetrievalSufficiency also needs expectations. The judge
is explicitly routed to a governed Databricks endpoint. Run this only after the
dataset, trace policy, model, index, and cost owner are approved. The connected
path uses the same fail-closed authorization helper as the application: Azure
uses an OData collection security filter, while Databricks standard endpoints
use an ARRAY filter. Storage-optimized Databricks indexes must expose a
platform-approved scalar ACL field before this lab can run. This evaluation
records two truthful evidence stages even when `candidate_k` equals `final_k`:
the scorer-visible top-level `retriever.final_context` `RETRIEVER` span contains
only current, supported documents supplied to the answer model after
deduplication, while the SDK's raw provider-candidate `retriever.search` span is
nested beneath it. The governed `predict_fn` span owns the complete invocation.
MLflow's evaluation harness can
otherwise enable OpenAI autologging temporarily, so this SDK-owned path disables
that second tracing owner before either evaluation begins.


In [ ]:
RUN_CONNECTED = False
connected_evaluation = None
if RUN_CONNECTED:
    import mlflow
    from agentic_ops_rag import OperationsRAGPipeline
    from mlflow.genai.scorers import (
        RetrievalGroundedness,
        RetrievalRelevance,
        RetrievalSufficiency,
        Safety,
    )

    from aai_core.tracing import TraceIntegration, traced

    if RUN_MLFLOW:
        raise RuntimeError(
            "Restart the kernel before connected evaluation: local SQLite tracing "
            "and connected tracing cannot share one process."
        )
    session.context.configure_tracing(integration=TraceIntegration.SDK)
    mlflow.openai.autolog(disable=True)
    resources = session.connected_components(allow_network=True)

    def generate_answer(question: str, retrieved) -> str:
        context = mlflow_documents(retrieved)
        response = resources["model"].generate(
            [
                {"role": "system", "content": "Answer only from supplied evidence."},
                {"role": "user", "content": f"{question}\nEvidence: {context}"},
            ],
            temperature=0.0,
        )
        return response.content

    connected_pipeline = OperationsRAGPipeline(
        resources["retriever"],
        answer_generator=generate_answer,
    )

    @traced(name="operations-rag.predict", span_type="CHAIN")
    def predict_fn(
        question: str,
        tenant_id: str,
        region: str,
        allowed_groups: list[str],
    ) -> str:
        result = connected_pipeline.invoke(
            question,
            tenant_id=tenant_id,
            region=region,
            allowed_groups=allowed_groups,
            mode="hybrid",
            candidate_k=3,
            final_k=3,
        )
        return result.answer

    judge_model = session.judge_model_uri()
    def evaluation_row(case):
        reference = pipeline.invoke(
            case.question,
            tenant_id=case.tenant_id,
            region=case.region,
            allowed_groups=case.allowed_groups,
            mode="hybrid",
        )
        return {
            "inputs": {
                "question": case.question,
                "tenant_id": case.tenant_id,
                "region": case.region,
                "allowed_groups": list(case.allowed_groups),
            },
            "expectations": {
                "expected_response": reference.answer,
                "expected_document_ids": list(case.expected_document_ids),
            },
        }

    policy_data = [
        evaluation_row(case)
        for case in cases
        if not case.answerable or case.expects_action_proposal
    ]
    rag_data = [
        evaluation_row(case)
        for case in cases
        if case.answerable and not case.expects_action_proposal
    ]
    connected_evaluation = {
        "policy": mlflow.genai.evaluate(
            data=policy_data,
            predict_fn=predict_fn,
            scorers=[Safety(model=judge_model)],
        ),
        "rag": mlflow.genai.evaluate(
            data=rag_data,
            predict_fn=predict_fn,
            scorers=[
                RetrievalRelevance(model=judge_model),
                RetrievalGroundedness(model=judge_model),
                RetrievalSufficiency(model=judge_model),
                Safety(model=judge_model),
            ],
        ),
    }
connected_evaluation


## Knowledge check

Answer from the evidence you produced, not from memory:

1. Which span output fields make retrieval visible to RAG judges?
2. Which checks must remain deterministic even when judges are available?
3. Why must one invocation have exactly one tracing owner?

<details>
<summary>How to use this check</summary>

If an answer cannot point to a row, trace, contract, or failed check from this
lesson, revisit the exercise before moving on.

</details>


## Recap

You validated retriever evidence, ran a deterministic release gate, and prepared
an explicit MLflow 3 judge path. Local fixtures never masquerade as provider
quality or cost evidence. Lesson 05 converts the same measurements into a
baseline, change, result, decision, and immutable application release.
